# 第 6 周练习：“价格合适”顶点项目

该笔记本实现了**价格预测**管道，可根据亚马逊数据从描述中估计产品价格。它结合了所有 5 天的学习内容：

- **第 1 天**：数据管理 — 加载、解析、重复数据删除、样本
- **第2天**：数据预处理——用LLM重写产品描述
- **第 3 天**：基线和传统 ML — 随机、常量、线性回归、词袋、随机森林、XGBoost
- **第 4 天**：深度学习和法学硕士 — 神经网络、法学硕士推理 (gpt-4.1-nano)
- **第 5 天**：微调 — 微调 GPT-4.1-nano 以进行价格预测

### 要求
- 从存储库根目录或 **week6** 目录运行（设置单元配置路径）
- 带有“HF_TOKEN”（HuggingFace）的“.env”以及用于 LLM/微调的可选“OPENAI_API_KEY”
- 使用 `LITE_MODE = True` 进行自由执行（20k 列，来自 Hub 的预处理数据）

＃＃ 设置

In [ ]:
# 将 week6 添加到路径中，以便定价包解析
import sys
import os
from pathlib import Path

cwd = Path.cwd()
# 查找 week6：cwd、父级（存储库根）或向上 2 级（来自社区贡献/名称）
for candidate in [cwd, cwd.parent, cwd.parent.parent]:
    week6_dir = candidate / "week6"
    if week6_dir.exists():
        break
else:
    week6_dir = cwd
sys.path.insert(0, str(week6_dir))
os.chdir(week6_dir)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(override=True)
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token, add_to_git_credential=True)
    print("HuggingFace login successful")
else:
    print("HF_TOKEN not set — add to .env to load datasets from Hub")

## 第一天：数据管理

加载原始产品数据，探索分布（价格、文本长度、类别），并使用 HuggingFace 中精选的数据集。原始数据：[McAuley-Lab/Amazon-Reviews-2023](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023)。

In [ ]:
LITE_MODE = True  # Use lite dataset (20k train) for fast, free runs
USERNAME = "ed-donner"

# 加载原始项目（带有全文）以进行探索
from pricer.items import Item
raw_dataset = f"{USERNAME}/items_raw_lite" if LITE_MODE else f"{USERNAME}/items_raw_full"
train_raw, val_raw, test_raw = Item.from_hub(raw_dataset)
items_raw = train_raw + val_raw + test_raw
print(f"Loaded {len(items_raw):,} raw items")

In [ ]:
# 第一天：探索数据
import matplotlib.pyplot as plt
from collections import Counter

prices = [item.price for item in items_raw]
lengths = [len(item.full or "") for item in items_raw]

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(prices, bins=50, color="blueviolet", rwidth=0.9)
plt.xlabel("Price ($)")
plt.ylabel("Count")
plt.title(f"Price distribution (avg ${sum(prices)/len(prices):.1f})")
plt.subplot(1, 2, 2)
plt.hist(lengths, bins=50, color="skyblue", rwidth=0.9)
plt.xlabel("Text length (chars)")
plt.ylabel("Count")
plt.title(f"Text length (avg {sum(lengths)/len(lengths):.0f})")
plt.tight_layout()
plt.show()

In [ ]:
# 类别计数
cat_counts = Counter([item.category for item in items_raw])
plt.figure(figsize=(10, 4))
plt.bar(cat_counts.keys(), cat_counts.values(), color="goldenrod")
plt.xticks(rotation=30, ha="right")
plt.title("Items per category")
plt.show()

## 第 2 天：数据预处理

使用法学硕士将产品描述重写为标准格式。这标准化了下游模型的数据。我们加载已经具有“summary”字段的预处理数据集（“items_lite”/“items_full”）。

In [ ]:
# 预处理系统提示（在第 2 天用于创建摘要）
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [ ]:
# 加载预处理项目（带有摘要）以进行建模
dataset = f"{USERNAME}/items_lite" if LITE_MODE else f"{USERNAME}/items_full"
train, val, test = Item.from_hub(dataset)
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

In [ ]:
# 示例：原始数据与预处理数据
print("Sample item summary (pre-processed):")
print(train[0].summary)

## 第 3 天：基线和传统机器学习

使用“pricer.evaluator”中的“evaluate”函数评估简单基线（随机、常量）和传统 ML 模型（线性回归、词袋、随机森林、XGBoost）。

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
import random
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate

In [ ]:
# 基线 1：随机猜测
random.seed(42)
def random_pricer(item):
    return random.randrange(1, 1000)

evaluate(random_pricer, test, size=100)

In [ ]:
# 基线 2：恒定（训练平均值）
train_avg = sum(item.price for item in train) / len(train)
def constant_pricer(item):
    return train_avg

evaluate(constant_pricer, test, size=100)

In [ ]:
# 简单模型的特征
def get_features(item):
    return {
        "weight": item.weight or 0,
        "weight_unknown": 1 if (item.weight or 0) == 0 else 0,
        "text_length": len(item.summary or ""),
    }

def list_to_df(items):
    fs = [get_features(i) for i in items]
    df = pd.DataFrame(fs)
    df["price"] = [i.price for i in items]
    return df

train_df = list_to_df(train)
test_df = list_to_df(test)
feature_cols = ["weight", "weight_unknown", "text_length"]

X_train, y_train = train_df[feature_cols], train_df["price"]
X_test, y_test = test_df[feature_cols], test_df["price"]

In [ ]:
# 手工制作特征的线性回归
np.random.seed(42)
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

def linear_pricer(item):
    f = get_features(item)
    pred = lr_model.predict(pd.DataFrame([f])[feature_cols])[0]
    return max(0, pred)

evaluate(linear_pricer, test, size=100)

In [ ]:
# 词袋+线性回归
documents = [item.summary for item in train]
prices_arr = np.array([float(item.price) for item in train], dtype=float)

np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words="english")
X_vec = vectorizer.fit_transform(documents)

bow_model = LinearRegression()
bow_model.fit(X_vec, prices_arr)

def bow_linear_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(0, bow_model.predict(x)[0])

evaluate(bow_linear_pricer, test, size=100)

In [ ]:
# 随机森林（精简模式下速度的子集）
subset = min(15_000, len(train))
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X_vec[:subset], prices_arr[:subset])

def rf_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

evaluate(rf_pricer, test, size=100)

In [ ]:
# XGBoost（可选 - 如果未安装则跳过）
try:
    import xgboost as xgb
    xgb_model = xgb.XGBRegressor(n_estimators=200, random_state=42, n_jobs=4, learning_rate=0.1)
    xgb_model.fit(X_vec, prices_arr)

    def xgb_pricer(item):
        x = vectorizer.transform([item.summary])
        return max(0, xgb_model.predict(x)[0])

    evaluate(xgb_pricer, test, size=100)
except ImportError:
    print("XGBoost not installed — skip with: pip install xgboost")

## 第 4 天：深度学习和法学硕士

在词袋特征上训练普通神经网络，然后使用前沿 LLM 进行零样本价格估计。

In [ ]:
# 神经网络
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

In [ ]:
# 用于二进制词袋的 HashingVectorizer
np.random.seed(42)
hv = HashingVectorizer(n_features=5000, stop_words="english", binary=True)
X_hv = hv.fit_transform(documents)
y_hv = np.array([float(i.price) for i in train], dtype=float)

X_t = torch.FloatTensor(X_hv.toarray())
y_t = torch.FloatTensor(y_hv).unsqueeze(1)
X_tr, X_vl, y_tr, y_vl = train_test_split(X_t, y_t, test_size=0.01, random_state=42)
loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)

In [ ]:
# 定义和训练神经网络
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.layers(x)

nn_model = NeuralNetwork(X_t.shape[1])
loss_fn = nn.MSELoss()
opt = optim.Adam(nn_model.parameters(), lr=0.001)

for epoch in range(2):
    nn_model.train()
    for bx, by in tqdm(loader):
        opt.zero_grad()
        loss = loss_fn(nn_model(bx), by)
        loss.backward()
        opt.step()

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
def nn_pricer(item):
    nn_model.eval()
    with torch.no_grad():
        x = hv.transform([item.summary])
        x = torch.FloatTensor(x.toarray())
        out = nn_model(x)[0].item()
    return max(0, out)

evaluate(nn_pricer, test, size=100)

In [ ]:
# LLM 零次价格预测（需要 OPENAI_API_KEY）
def messages_for(item):
    msg = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": msg}]

def gpt_nano_pricer(item):
    from litellm import completion
    r = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return r.choices[0].message.content

# 取消注释运行（使用 API 积分）
# 评估（gpt_nano_pricer，测试，大小= 50）

## 第 5 天：微调前沿模型

在一小部分（产品摘要、价格）示例上微调 GPT-4.1-nano，以改进价格预测。需要“OPENAI_API_KEY”并产生 API 费用。

In [ ]:
# 准备微调数据（JSONL格式）
import json

def ft_messages_for(item):
    msg = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": msg},
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]

def make_jsonl(items):
    lines = []
    for item in items:
        msgs = ft_messages_for(item)
        lines.append(json.dumps({"messages": msgs}))
    return "\n".join(lines)

# 使用小子集（OpenAI 建议 50-100 个示例）
ft_train = train[:100]
ft_val = val[:50]

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
import os
os.makedirs("jsonl", exist_ok=True)
with open("jsonl/fine_tune_train.jsonl", "w") as f:
    f.write(make_jsonl(ft_train))
with open("jsonl/fine_tune_val.jsonl", "w") as f:
    f.write(make_jsonl(ft_val))
print("JSONL files written")

In [ ]:
# 上传并微调（仅在设置了 OPENAI_API_KEY 并且您希望产生费用时运行）
# 从 openai 导入 OpenAI
# 客户端 = OpenAI()
# 将 open("jsonl/fine_tune_train.jsonl", "rb") 作为 f：
# train_file = client.files.create(file=f,目的=“微调”)
# 使用 open("jsonl/fine_tune_val.jsonl", "rb") 作为 f：
# val_file = client.files.create(file=f,目的=“微调”)
# 工作 = client.fine_tuning.jobs.create(
# 训练文件=训练文件.id,
# valid_file=val_file.id,
# 型号=“gpt-4.1-nano-2025-04-14”，
# 种子=42，
# 超参数={“n_epochs”：1，“batch_size”：1}，
# 后缀=“定价者”，
# )
# 工作 ID = 工作 ID
# print(f"微调作业：{job_id}")

In [ ]:
# 作业完成后，使用微调模型：
# fine_tuned = client.fine_tuning.jobs.retrieve(job_id).fine_tuned_model
# def ft_pricer(项目):
# r = client.chat.completions.create(
# 模型=微调，
# messages=[{"role": "user", "content": f"预估价格...\n\n{item.summary}"}],
# 最大令牌数=7,
#     )
# 返回 r.choices[0].message.content
# 评估（ft_pricer，测试）

## 总结

该顶点涵盖：
- **第一天**：来自亚马逊评论的数据管理
- **第 2 天**：基于 LLM 的标准化描述预处理
- **第 3 天**：基线（随机、恒定）和传统 ML（线性、RF、XGBoost）
- **第 4 天**：神经网络和零样本 LLM 推理
- **第 5 天**：微调 GPT-4.1-nano 以进行价格预测

设置“LITE_MODE = False”并使用完整数据集以获得更好的模型性能。使用 20k 个示例进行微调可以实现约 68 美元的平均误差。